# 03. Hierarchical observations and sampling

![Hierarchical groups and balanced estimation](../images/03_hierarchical_observations.svg)

Lessons 01 and 02 shaped and compared clips. This notebook asks where the clips came
from, because a row only becomes evidence once you say which population it represents.

**What you will do:** compute spread and empirical distributions, simulate dependent
groups, expose pseudoreplication, compare row and group estimands, sample conditionally,
and build group-disjoint partitions.

The [lecture](../lectures/03_hierarchical_observations.md) motivates each step; the cells
here make the consequences measurable.

In [ ]:
import random
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

SEED = 19
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print(f'numpy={np.__version__}, synthetic data only')

## 1. Means imply weights

Every average carries a weighting rule, whether or not you chose it. The row mean gives
group $g$ the weight $n_g/N$, which is just the share of rows that group happened to
contribute. The group-balanced mean averages within each group first and then gives every
group the weight $1/G$.

Both are correct arithmetic and they answer different population questions: one about a
randomly chosen recording, the other about a randomly chosen participant. Below, two
rows of zeros and eight rows of ten make the gap impossible to miss.

In [ ]:
values = np.array([0., 0., 10., 10., 10., 10., 10., 10., 10., 10.])
groups = np.array([0, 0] + [1] * 8)
row_mean = values.mean()
group_means = np.array([values[groups == g].mean() for g in np.unique(groups)])
balanced_mean = group_means.mean()
assert row_mean == 8.0 and balanced_mean == 5.0
print(f'row-weighted={row_mean:.1f}, group-balanced={balanced_mean:.1f}')

## 2. Variance, standard deviation, and quantiles

An estimate needs a companion measure of spread. Population variance divides squared
deviations by $N$. Sample variance divides by $N-1$ (`ddof=1`) to remove the bias created
by estimating the center from the same data, and standard deviation returns to the
original units.

Quantiles describe the same data by rank instead of by squared distance, which makes them
far less sensitive to a single extreme value. Watch how the value 100 in the sample below
pulls the mean and standard deviation while barely moving the median.

In [ ]:
sample = np.array([1., 2., 2., 3., 100.])
population_var = np.var(sample, ddof=0)
sample_var = np.var(sample, ddof=1)
sample_std = np.std(sample, ddof=1)
quartiles = np.quantile(sample, [0.25, 0.5, 0.75], method='linear')
assert sample_var > population_var
assert np.isclose(sample_std ** 2, sample_var)
assert quartiles[1] == 2.0
print('sample variance/std:', round(sample_var, 2), round(sample_std, 2))
print('quartiles:', quartiles)

## 3. An empirical cumulative distribution

Means and quantiles are single summaries. The ECDF describes the whole distribution
without assuming a shape: at threshold $a$ it reports the observed fraction of values at
or below $a$, so it rises from zero to one in steps.

Computed naively it compares every value against every threshold. Sorting once and using
`np.searchsorted(..., side='right')` replaces those comparisons with a binary search,
which matters as soon as there are many thresholds. The result must be non-decreasing,
and the assertions check exactly that.

In [ ]:
def ecdf(values, thresholds):
    sorted_values = np.sort(np.asarray(values))
    thresholds = np.asarray(thresholds)
    return np.searchsorted(sorted_values, thresholds, side='right') / len(sorted_values)

thresholds = np.array([1., 2., 3., 100.])
probabilities = ecdf(sample, thresholds)
np.testing.assert_allclose(probabilities, [0.2, 0.6, 0.8, 1.0])
assert np.all(np.diff(probabilities) >= 0)
print(list(zip(thresholds, probabilities)))

## 4. Simulate clustered dependence

So far every row counted the same. Rows from one participant usually resemble each other,
and the simplest model of why is $y_{gi}=\mu+a_g+e_{gi}$.

Read it as three sources: $\mu$ is the center shared by everyone, $a_g$ shifts all rows
in group $g$ together, and $e_{gi}$ jostles each row on its own. Two rows in one group
share the same $a_g$, so they are correlated. The intraclass correlation
$\rho=\sigma_a^2/(\sigma_a^2+\sigma_e^2)$ is the share of total variance that comes
from between-group differences, and the simulation below fixes it by construction.

In [ ]:
G, rows_per_group = 40, 20
sigma_group, sigma_noise = 2.0, 1.0
group_effect = rng.normal(0, sigma_group, size=G)
group_id = np.repeat(np.arange(G), rows_per_group)
outcome = 5.0 + group_effect[group_id] + rng.normal(0, sigma_noise, size=G*rows_per_group)
rho_theory = sigma_group**2 / (sigma_group**2 + sigma_noise**2)
assert outcome.shape == group_id.shape == (800,)
assert np.unique(group_id).size == G
print(f'rows={len(outcome)}, groups={G}, theoretical ICC={rho_theory:.2f}')

## 5. Pseudoreplication changes uncertainty

Dependence does not move the estimate. It widens the interval you are entitled to report.

With equal group size $m$, the design effect is about $1+(m-1)\rho$, and dividing the row
count by it gives a rough effective sample size. A naive row-level standard error pretends
all 800 rows are independent, while a standard error built from the 40 group means
respects the units that were actually sampled. The assertion below states the expected
direction: the group-level standard error must be the larger of the two.

In [ ]:
naive_se = outcome.std(ddof=1) / np.sqrt(len(outcome))
_, inverse = np.unique(group_id, return_inverse=True)
sums = np.bincount(inverse, weights=outcome)
counts = np.bincount(inverse)
observed_group_means = sums / counts
group_se = observed_group_means.std(ddof=1) / np.sqrt(G)
design_effect = 1 + (rows_per_group - 1) * rho_theory
effective_n = len(outcome) / design_effect
assert group_se > naive_se
print(f'naive row SE={naive_se:.3f}, group SE={group_se:.3f}, effective N~{effective_n:.1f}')

## 6. Conditional sampling changes group probabilities

The same distinction appears when drawing data rather than summarizing it. Sampling rows
directly selects group $g$ with probability $n_g/N$, so a group with more rows shows up
more often. Two-stage sampling picks a group uniformly and then a row inside it, giving
every group probability $1/G$.

With two groups holding 2 and 8 rows, the first scheme should visit group 1 about 80
percent of the time and the second about 50 percent. Writing the sampler is often the
fastest way to pin down an ambiguous estimand.

In [ ]:
unequal_groups = np.array([0]*2 + [1]*8)
row_draw_groups = unequal_groups[rng.integers(0, len(unequal_groups), size=20_000)]
two_stage_groups = rng.integers(0, 2, size=20_000)
row_p_group1 = np.mean(row_draw_groups == 1)
balanced_p_group1 = np.mean(two_stage_groups == 1)
assert abs(row_p_group1 - 0.8) < 0.02
assert abs(balanced_p_group1 - 0.5) < 0.02
print(f'P(group 1): row sampling={row_p_group1:.3f}, two-stage={balanced_p_group1:.3f}')

## 7. Split whole groups, not related rows

Weighting and uncertainty both hinge on the unit, and so does evaluation. If one clip
from a participant trains the model and another clip from that same participant tests it,
the score measures recognition of that person rather than generalization to new people.

`GroupShuffleSplit` keeps every group's rows on a single side of the split. Passing a
group identifier is necessary but not sufficient, so verify the realized group sets: the
assertion below requires the train and test group sets to be disjoint and every row to be
accounted for.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(splitter.split(outcome[:, None], groups=group_id))
train_groups = set(group_id[train_idx])
test_groups = set(group_id[test_idx])
assert train_groups.isdisjoint(test_groups)
assert len(train_idx) + len(test_idx) == len(outcome)
print(f'train groups={len(train_groups)}, test groups={len(test_groups)}, overlap=0')

## 8. Support is not exposure or independent evidence

One more triple keeps a sampling policy honest, and it returns in later lessons.
**Support** is the set of distinct units a policy could visit. **Exposure** is the number
of draws made, including repeats. **Realized support** counts the distinct units actually
visited.

Drawing 20 times with replacement from 4 sequence-window pairs gives exposure 20 and
realized support of at most 4. Raising exposure can uncover more of the support set, and
it never enlarges it. Note also that the 4 pairs come from only 2 sequences, so a support
unit is not automatically an independent participant.

In [ ]:
support = np.array(['seq-1@0', 'seq-1@8', 'seq-2@0', 'seq-2@8'])
exposure = 20
drawn_units = rng.choice(support, size=exposure, replace=True)
realized_support = np.unique(drawn_units).size
assert exposure == len(drawn_units)
assert 1 <= realized_support <= len(support)
assert len(support) == 4 and len({unit.split('@')[0] for unit in support}) == 2
print(f'support={len(support)}, exposure={exposure}, realized support={realized_support}')

## Exercises and final takeaways

**Efficiency.** Encode group labels once with `np.unique(..., return_inverse=True)` and
summarize with `np.bincount`, which needs one pass instead of one Boolean scan per group.

**Exercises:** (1) Change the group and noise standard deviations, and predict the ICC and
design effect before running. (2) Create unequal group sizes and compare row, equal-group,
and custom weighted means. (3) Repeat the group split several times and summarize the
test-group counts.

**Takeaways:** descriptive statistics encode weights, so name the estimand first; repeated
rows are not automatically independent evidence; the experimental unit governs both
uncertainty and splitting; vectorized group summaries are clearer and faster than
per-group scans.

## Continue learning

[Previous notebook: 02](02_inner_product_geometry.ipynb) | [Lecture](../lectures/03_hierarchical_observations.md) | [Curriculum](../README.md) | [Next notebook: 04](04_attention_and_positions.ipynb)